# Database Access

Python's `sqlite3` module provides a zero-configuration relational database in the stdlib. For larger projects or different backends, SQLAlchemy adds a unified API, a SQL expression language, and a full ORM, all without changing your application logic when you switch databases.

**What's inside:** `sqlite3` (connect, execute, cursor, context manager, parameterised queries), SQLAlchemy Core (DDL, insert, select), and the SQLAlchemy ORM (declarative models, session, queries).

**Learn more:** [sqlite3](https://docs.python.org/3/library/sqlite3.html) · [SQLAlchemy](https://docs.sqlalchemy.org)

## Setup

In [ ]:
%pip install sqlalchemy

## 1. sqlite3: basics

In [ ]:
import sqlite3

# :memory: creates an in-process database that vanishes when the connection closes
con = sqlite3.connect(':memory:')
cur = con.cursor()

cur.execute('''
    CREATE TABLE users (
        id    INTEGER PRIMARY KEY AUTOINCREMENT,
        name  TEXT NOT NULL,
        email TEXT UNIQUE NOT NULL,
        age   INTEGER
    )
''')

cur.executemany(
    'INSERT INTO users (name, email, age) VALUES (?, ?, ?)',
    [
        ('Alice', 'alice@example.com', 32),
        ('Bob',   'bob@example.com',   25),
        ('Carol', 'carol@example.com', 28),
    ]
)
con.commit()

for row in cur.execute('SELECT * FROM users ORDER BY age'):
    print(row)

con.close()

## 2. sqlite3 as a context manager

In [ ]:
import sqlite3

# The connection itself works as a context manager: commits on success, rolls back on error
with sqlite3.connect(':memory:') as con:
    con.execute('CREATE TABLE items (id INTEGER PRIMARY KEY, name TEXT, qty INTEGER)')
    con.execute('INSERT INTO items VALUES (1, "apples", 50)')
    con.execute('INSERT INTO items VALUES (2, "bananas", 30)')
    # con.commit() is called automatically on __exit__

    rows = con.execute('SELECT name, qty FROM items WHERE qty > ?', (20,)).fetchall()
    print(rows)

## 3. Row factory: access columns by name

In [ ]:
import sqlite3

con = sqlite3.connect(':memory:')
con.row_factory = sqlite3.Row   # rows behave like dicts

con.execute('CREATE TABLE products (id INTEGER PRIMARY KEY, name TEXT, price REAL)')
con.executemany('INSERT INTO products VALUES (?,?,?)', [
    (1, 'Widget', 9.99), (2, 'Gadget', 24.99), (3, 'Doohickey', 4.99)
])

for row in con.execute('SELECT * FROM products ORDER BY price DESC'):
    print(f"{row['name']:12}  ${row['price']:.2f}")

con.close()

## 4. SQLAlchemy Core: SQL expression language

In [ ]:
from sqlalchemy import create_engine, MetaData, Table, Column, Integer, String, Float, insert, select

engine = create_engine('sqlite:///:memory:', echo=False)
meta = MetaData()

products = Table('products', meta,
    Column('id',    Integer, primary_key=True),
    Column('name',  String(100), nullable=False),
    Column('price', Float, nullable=False),
    Column('stock', Integer, default=0),
)
meta.create_all(engine)

with engine.begin() as con:
    con.execute(insert(products), [
        {'name': 'Widget',    'price': 9.99,  'stock': 100},
        {'name': 'Gadget',    'price': 24.99, 'stock': 50},
        {'name': 'Doohickey', 'price': 4.99,  'stock': 200},
    ])

with engine.connect() as con:
    stmt = select(products).where(products.c.price < 15).order_by(products.c.price)
    for row in con.execute(stmt):
        print(row._mapping)

## 5. SQLAlchemy ORM: declarative models

In [ ]:
from sqlalchemy import create_engine, String, Integer, Float, ForeignKey
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, Session

engine = create_engine('sqlite:///:memory:', echo=False)

class Base(DeclarativeBase):
    pass

class Category(Base):
    __tablename__ = 'categories'
    id:   Mapped[int] = mapped_column(Integer, primary_key=True)
    name: Mapped[str] = mapped_column(String(50))
    products: Mapped[list['Product']] = relationship(back_populates='category')

class Product(Base):
    __tablename__ = 'products'
    id:          Mapped[int]   = mapped_column(Integer, primary_key=True)
    name:        Mapped[str]   = mapped_column(String(100))
    price:       Mapped[float] = mapped_column(Float)
    category_id: Mapped[int]   = mapped_column(ForeignKey('categories.id'))
    category:    Mapped[Category] = relationship(back_populates='products')

Base.metadata.create_all(engine)

with Session(engine) as session:
    elec = Category(name='Electronics')
    session.add(elec)
    session.flush()   # assign elec.id without committing

    session.add_all([
        Product(name='Laptop',     price=999.99,  category=elec),
        Product(name='Mouse',      price=29.99,   category=elec),
        Product(name='Keyboard',   price=79.99,   category=elec),
    ])
    session.commit()

with Session(engine) as session:
    results = session.query(Product).filter(Product.price < 100).order_by(Product.price).all()
    for p in results:
        print(f'{p.name:12}  ${p.price:.2f}  [{p.category.name}]')

## 6. ORM: update and delete

In [ ]:
from sqlalchemy.orm import Session

with Session(engine) as session:
    # update
    mouse = session.query(Product).filter_by(name='Mouse').one()
    mouse.price = 24.99
    session.commit()
    print('updated:', mouse.name, mouse.price)

    # delete
    keyboard = session.query(Product).filter_by(name='Keyboard').one()
    session.delete(keyboard)
    session.commit()

    remaining = session.query(Product).order_by(Product.price).all()
    for p in remaining:
        print(p.name, p.price)